# Week 07: Attention, GPT and applications

## Text as Data

Professor: Elliott Ash, NYU

TA: Eduardo Zago, NYU

Last class we went over (finalizing) pre-processing of text into tokens, embeddings (context and positional) and into a basic attention mechanism.

Objectives:

1) Enhanced Self-Attention
2) Causal Self-Attention
3) Multi-head Attention
4) All together: GPT-2
5) Apps: BERTopic and DeepLatent

Code adapted from Raschka Ch: 3.4-3.6

Also see:



* Andrej Karpathy's [micro-gpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)
*  An [explanation](https://www.towardsdeeplearning.com/andrej-karpathy-just-built-an-entire-gpt-in-243-lines-of-python-7d66cfdfa301)
* And this [blog](https://jaykmody.com/blog/gpt-from-scratch)









In [1]:
import collections
from collections import Counter
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from string import punctuation
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# set random seed
# We upgrade torch, torchvision, and torchaudio together to avoid CUDA version mismatches
!pip install torch torchvision torchaudio --upgrade --force-reinstall
!pip install transformers --force-reinstall
!pip install gensim
!pip install tiktoken

nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
from nltk.corpus import stopwords
stoplist = set(stopwords.words('english'))

from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached filelock-3.29.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached cuda_bindings-13.3.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5

  Using cached transformers-5.12.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.26.7-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached tqdm-4.68.2-py3-none-any.whl.metadata (58 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.29.3-p

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Last class we saw the simple attention mechanism. Main idea: for each query, we need to understand how the other tokens in the context window relate to it. We do this by computing similarities, normalizing and weighting the initial outputs with respect to a certain query. Steps:

1) For $j\leq i$, compute $sim(x_i, x_j) = x_i^{T}x_j$
2) Normalize to get attention weights: $\alpha_{ij} = (softmax([sim(x_i, x_1), ..., sim(x_i, x_j)])_j$
3) Compute attention scores for each query: $a_i = \sum_{j \leq i} \alpha_{ij}x_j$


In [2]:
# Keep the first sentence:

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = attn_weights @ inputs

print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### Enhanced self-attention

Notice that the same word can play three different roles in this process:

1) When it is the current word $i$ in calculating $sim(x_i, x_j)$.
2) When it is surrounding word $j$ in calculating $sim(x_i, x_j)$
3) When it is surrounding word $j$ in calculcating $\sum_{j \leq i} \alpha_{ij}x_j$

Thus we will use three different vector to represent each word in a different role, and add parameters that our network can learn:

1) query ($q_i = W_q x_i$)
2) key ($k_i = W_k x_i$)
3) value ($v_i = W_v x_i$).

In [3]:
x_2 = inputs[1]  # As last exercise, second token is the query
d_in = inputs.shape[1]  # Input embedding size
d_out = 2  # output (different from input for didactic reasons)

# Initialize the weight (parameter) matrices:

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Compute query, key and value vectors

query_2 = x_2 @ W_query # current item
key_2 = x_2 @ W_key # indexing and searching
value_2 = x_2 @ W_value # actual content
print(query_2)

tensor([0.4306, 1.4551])


Compute similarities as we did before, with respect to the query:

$$sim(x_i, x_j) = \frac{q_i^{T} k_j}{\sqrt{d_k}}$$

In [4]:
# We need al keys and values weights to compute the attention score of the second input

keys = inputs @ W_key
values = inputs @ W_value

d_k = keys.shape[-1]

attn_scores_2 = query_2 @ keys.T / d_k**0.5  # scaled dot product (to avoid backpropagation issues, such as really small gradients)
print(attn_scores_2)

tensor([0.8984, 1.3098, 1.2806, 0.7633, 0.3944, 1.0918])


Normalize them:

$\alpha_{ij} = (softmax([sim(x_i, x_1), ..., sim(x_i, x_j)])_j$

In [5]:
# Attention scores to attention weights:

attn_weights_2 = torch.softmax(attn_scores_2, dim=-1) #
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Compute the final context vector:

$\sum_{j \leq i} \alpha_{ij}v_j$

In [6]:
# Context vectors:

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


Compacting it all:

In [8]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # Use transpose(-2, -1) to support both 2D and 3D (batched) tensors
        attn_scores = queries @ keys.transpose(-2, -1)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


### Causal attention mechanism

For decoder tasks, we will want the self-attention mechanisms to only consider tokens that appear prior to the current position.

In [9]:
queries = sa_v2.W_query(inputs)     #1
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

context_length = attn_scores.shape[0]
###

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

###

attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)
tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000

In [10]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


batch = torch.stack((inputs, inputs), dim=0)
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [11]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                 d_in, d_out, context_length, dropout, qkv_bias
             )
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Next week: GPT2 IMPLEMENTATION!! (Finally).

For now we can look at some applications of the embeddings we have generated, as well as other tools we have used before:

### BERTOpic

<div style="text-align: center;">
<img src='https://maartengr.github.io/BERTopic/img/algorithm.png' width='600'>
</div>

In [5]:
!pip install bertopic[visualization]
from bertopic import BERTopic

In [13]:
from google.colab import files
uploaded_2 = files.upload()

Saving death-penalty-cases.csv to death-penalty-cases.csv


In [6]:
df1 = pd.read_csv('death-penalty-cases.csv')
docs = df1['snippet'].tolist()
len(docs)

32567

In [7]:
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
# Processing the full 'docs' list (all 32,567 rows)
topics, probs = topic_model.fit_transform(docs)

# Explore topics
print(topic_model.get_topic_info().head(10))

2026-06-12 17:31:17,314 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1018 [00:00<?, ?it/s]

2026-06-12 17:31:54,746 - BERTopic - Embedding - Completed ✓
2026-06-12 17:31:54,748 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-12 17:32:53,258 - BERTopic - Dimensionality - Completed ✓
2026-06-12 17:32:53,261 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-12 17:37:50,576 - BERTopic - Cluster - Completed ✓
2026-06-12 17:37:50,599 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-12 17:37:51,965 - BERTopic - Representation - Completed ✓


   Topic  Count                                         Name  \
0     -1  13265                      -1_the_to_death_penalty   
1      0    611                        0_she_her_about_could   
2      1    535  1_florida_unconstitutional_statute_floridas   
3      2    490         2_published_nondeath_felony_november   
4      3    421            3_degree_first_murder_firstdegree   
5      4    378            4_hearing_eligible_found_preclude   
6      5    370     5_sanctions_sanction_discovery_quotdeath   
7      6    366                   6_notice_intent_filed_seek   
8      7    350          7_stat_antiterrorism_effective_publ   
9      8    306    8_appellant_appellants_assessed_convicted   

                                      Representation  \
0  [the, to, death, penalty, in, was, that, not, ...   
1  [she, her, about, could, vote, stated, juror, ...   
2  [florida, unconstitutional, statute, floridas,...   
3  [published, nondeath, felony, november, octobe...   
4  [degree, fir

In [8]:
!pip install "numpy<2.1" umap-learn

# NOTE: You MUST restart the runtime after installing these packages.
# Go to: Runtime -> Restart Session

# Visualizations:
import umap
topic_model.visualize_topics()                        # intertopic distance map

In [28]:
topic_model.visualize_barchart(top_n_topics=8)        # top words per topic

In [29]:
topic_model.visualize_heatmap()

In [30]:
# ── 5. Assign topics back to full dataframe ───────────────────────────────────────
df1['topic'] = topics
df1['topic_label'] = df1['topic'].apply(
    lambda t: " | ".join([w for w, _ in topic_model.get_topic(t)[:3]]) if t != -1 else "outlier"
)

print(f"Assigned topics to all {len(df1)} rows.")
display(df1[['snippet', 'topic', 'topic_label']].head())

Assigned topics to all 32567 rows.


,snippet,topic,topic_label
0,N.J. ( )\n A. d \nIN RE WAIVER OF DEATH PE...,-1,outlier
1,"whether the death penalty is, per se, unconsti...",-1,outlier
2,# ;s contention that the assessment of the dea...,-1,outlier
3,. d ( )\n -NMSC- \nIN THE MATTER OF DEATH PE...,-1,outlier
4,assume the district attorney orally waived the...,-1,outlier


### DeepLatent:

Based on: [DeepLatent](https://github.com/PinchOfData/DeepLatent). Topic modelling is a way of extracting latent topics from unstructured text. However, there are a few things to have in mind while doing this task:

What are covariates in a topic model?
Two types:
1) prevalence: a covariate that affects HOW MUCH each topic appears in a doc   e.g. "does the year of the case change which topics dominate?"

2) content:    a covariate that affects THE WORDS used to discuss a topic e.g. "do defense lawyers and prosecutors use different words for the same topic?"

Example: imagine two death-penalty cases about "mental illness".

prevalence covariate → older cases talk less about mental illness overall content covariate    → prosecution says "dangerous", defense says "disorder"

In [4]:
!pip install deeplatent

In [9]:
# FIX: Downgrade numpy to satisfy numba requirements
# After running this line for the first time, you MUST go to
# Runtime -> Restart Session to apply the changes.
!pip install "numpy<2.1" --quiet

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from deeplatent.corpus import Corpus
from deeplatent.models import GTM

In [10]:
# 1. Define and fit the vectorizer using the full dataset
docs = df1['snippet'].tolist()

vectorizer = CountVectorizer(
    ngram_range=(1, 1),
    max_df=0.95,
    min_df=0.001,
    stop_words="english",
    token_pattern=r"(?u)\b\w{3,}\b"  # only words with 3 or more letters
)

vectorizer.fit(docs)

print("Number of words in the vocabulary: {}".format(len(vectorizer.get_feature_names_out())))

# 2. Define modalities using the external vectorizer
modalities = {
    "text": {
        "column": "snippet",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer
            }
        }
    }
}

# 3. Create GTMCorpus with prevalence and content covariates
train_dataset = Corpus(
    df=df1,
    modalities=modalities,
    prevalence="~ C(year)",
    content="~ C(state)"
)

Number of words in the vocabulary: 1863


In [11]:
from deeplatent import GTM

encoder_args = {
    "text_bow": {
        "hidden_dims": [256,128],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

decoder_args = {
    "text_bow": {
        "hidden_dims": [128,256],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

tm = GTM(
    train_dataset,
    n_topics=20,
    ae_type="wae", # WAE has very stable training and can use a Dirichlet prior, VAE has stronger theoretical guarantees but is prone to posterior collapse
    doc_topic_prior="logistic_normal",
    update_prior=True,
    encoder_args=encoder_args,
    decoder_args=decoder_args,
    batch_size=64,
    w_prior=1,
    return_best_model=True,
    num_steps=2000,
    print_every_n_steps=1000,
    print_topics=True,
    print_covariates=True
)

# Resume training if you think the topics need more refining
# tm.num_steps = tm.steps + 5000  # Train 5000 more steps
# tm.train(train_data)

Step   1000	Mean Training Loss:6.1421952
Rec Loss:5.9796743
Divergence Loss:0.1625210
Pred Loss:0.0000000

Topic_0: ['quot', 'cases', 'effective', 'act', 'case']
Topic_1: ['state', 'seek', 'court', 'murder', 'trial']
Topic_2: ['quot', 'jury', 'case', 'impose', 'court']
Topic_3: ['state', 'seek', 'cases', 'case', 'quot']
Topic_4: ['cases', 'state', 'murder', 'court', 'seek']
Topic_5: ['quot', 'vote', 'impose', 'juror', 'jurors']
Topic_6: ['jury', 'imposition', 'court', 'trial', 'impose']
Topic_7: ['state', 'seek', 'murder', 'cases', 'court']
Topic_8: ['quot', 'case', 'state', 'jury', 'trial']
Topic_9: ['court', 'jury', 'state', 'trial', 'seek']
Topic_10: ['court', 'jury', 'imposition', 'trial', 'imposed']
Topic_11: ['court', 'imposition', 'unconstitutional', 'statute', 'imposed']
Topic_12: ['murder', 'court', 'imposed', 'degree', 'unconstitutional']
Topic_13: ['imposition', 'statute', 'jury', 'aggravating', 'circumstances']
Topic_14: ['statute', 'imposition', 'unconstitutional', 'court'

In [12]:
print(
    "\n".join(
        [
            "{}: {}".format(str(k), str(v))
            for k, v in tm.get_topic_words(topK=5).items()
        ]
    )
)

Topic_0: ['cases', 'quot', 'case', 'court', 'effective']
Topic_1: ['murder', 'trial', 'state', 'jury', 'court']
Topic_2: ['quot', 'case', 'impose', 'jury', 'court']
Topic_3: ['seek', 'state', 'notice', 'intent', 'trial']
Topic_4: ['state', 'murder', 'life', 'imprisonment', 'case']
Topic_5: ['scruples', 'conscientious', 'jurors', 'religious', 'expressed']
Topic_6: ['jury', 'trial', 'imposition', 'court', 'aggravating']
Topic_7: ['state', 'seek', 'murder', 'life', 'guilty']
Topic_8: ['trial', 'jury', 'state', 'seek', 'defendant']
Topic_9: ['cases', 'quot', 'court', 'case', 'imposed']
Topic_10: ['court', 'quot', 'cases', 'imposition', 'case']
Topic_11: ['court', 'imposition', 'imposed', 'murder', 'state']
Topic_12: ['court', 'supreme', 'imposition', 'imposed', 'cases']
Topic_13: ['imposition', 'statute', 'circumstances', 'jury', 'arbitrary']
Topic_14: ['statute', 'imposition', 'unconstitutional', 'scheme', 'constitutionality']
Topic_15: ['quot', 'case', 'counsel', 'said', 'defendant']
Top

In [13]:
print("\nTop words per topic for a California case:")
print(pd.DataFrame(tm.get_topic_words(l_content_covariates = ["Intercept", "C(state)[T.CA]"], topK=5)).T)


Top words per topic for a California case:
                  0          1              2              3           4
Topic_0        quot      cases           case      effective         act
Topic_1   defendant     murder          trial           jury       court
Topic_2        quot  defendant         impose           case        jury
Topic_3        seek  defendant           life           case       state
Topic_4        life     murder           case   imprisonment   defendant
Topic_5      jurors       vote           quot         impose         law
Topic_6   defendant       jury          trial          court      impose
Topic_7   defendant     murder           seek           life       trial
Topic_8   defendant       jury          trial           seek        case
Topic_9        quot      cases           case          court   defendant
Topic_10       quot      court          cases      defendant        case
Topic_11  defendant      court        imposed         murder  imposition
Topic_1